# Predicting Student Dropout and Academic Success

### A comparative machine-learning study

**Author:** Al Christian Gobres
**Programme / Course:** Postgraduate Diploma in AI and Machine Learning  
**Date:** September 2026

**Objective:** Classify student outcomes as Dropout, Enrolled, or Graduate.

**Scope:** The model uses information through the second semester.
Enrollment-time prediction requires a different feature set.

## Dataset and Prediction Task

- **4,424 students** and **36 original predictors**
- Academic, demographic, financial, and economic information
- Three outcome classes with unequal frequencies

| Outcome | Students | Share |
|---|---:|---:|
| Dropout | 1,421 | 32.1% |
| Enrolled | 794 | 17.9% |
| Graduate | 2,209 | 49.9% |

Source: [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)

**Evaluation implication:** Macro F1 gives equal weight to each class.

Enrolled is an observed status; it does not establish the student's
eventual graduation or dropout outcome.

## Academic Performance Separates the Outcomes

Median values from the exploratory analysis:

| Feature | Dropout | Enrolled | Graduate |
|---|---:|---:|---:|
| First-semester approved units | 2 | 5 | 6 |
| Second-semester approved units | 0 | 4 | 6 |
| Second-semester grade | 0 | 12 | 13 |
| Age at enrollment | 23 | 20 | 19 |

- Semester performance shows clearer differences than entry grades.
- Enrolled students overlap with both other outcome groups.
- First- and second-semester approved units are strongly correlated:
  **Spearman ρ = 0.892**.
- Zero-heavy distributions make automatic IQR-based deletion inappropriate.

**These are associations, not causal effects.**

## Preprocessing and Feature Engineering

| Numerical predictors | Categorical predictors |
|---|---|
| Median imputation | Explicit missing category |
| Standard scaling | One-hot encoding |
| Valid zeros and extreme values retained | Rare categories grouped using training frequencies |

**Target:** Encoded separately and excluded from predictor transformations.

**Engineered features**

- Approved-to-enrolled unit ratios
- Units-without-evaluations ratios
- Zero-approved and zero-enrollment indicators
- Semester changes in grades and approved units
- Fixed age groups

**Quality checks:** Removed redundant automatic missingness indicators.
Retained one ratio above 1 pending clarification of the source definitions.

## Evaluation Design

| Partition | Students | Purpose |
|---|---:|---|
| Training | 2,654 | Five-fold stratified CV and tuning |
| Validation | 885 | Diagnostic evaluation |
| Test | 885 | Final evaluation after model selection |

**Selection criterion:** Highest mean training-CV macro F1.

- Fit preprocessing, feature selection, and PCA inside each CV fold.
- Compare full features, embedded selection, and PCA representations.
- Refit the selected pipeline on training plus validation data.
- Evaluate the fixed final model on the test set.

**Limitations:** Earlier EDA used the full dataset. Random splitting does
not establish performance on future cohorts.

## Random Forest and Logistic Regression Perform Similarly

| Best representation per algorithm | CV macro F1 | Validation macro F1 |
|---|---:|---:|
| Random forest — Full | **0.710** | **0.729** |
| Logistic regression — Full | 0.708 | 0.725 |
| RBF SVM — Full | 0.704 | 0.691 |
| Gradient boosting — Full | 0.696 | 0.713 |
| Dummy baseline | 0.222 | 0.222 |

**Algorithm rationale:** Linear baseline, nonlinear tree ensembles,
and an RBF decision boundary.

**Representation finding:** Full features achieved the highest CV score
within every evaluated algorithm.

**Selection:** Random forest won under the predefined rule.
Its 0.002 lead over logistic regression does not establish statistical superiority.

## Final Random Forest Results

**200 trees · maximum depth 12 · minimum leaf size 5 · balanced class weights**

| Test metric | Result |
|---|---:|
| Macro F1 | **0.720** |
| Accuracy | **76.2%** |
| Dropout precision | **86.0%** |
| Dropout recall | **73.2%** |

**Main limitation:** Enrolled F1 = **0.526**.

![Final test confusion matrices](../reports/reference/figures/final_test_confusion_matrices.png)

**208 of 284 dropouts identified; 76 missed.**  
**96 of 159 Enrolled students correctly classified.**

## Academic Progress Strongly Influences Predictions

- Average predicted dropout probability declines markedly as
  second-semester approved units increase.
- First-semester approved units show a smaller response.
- Age has a comparatively modest average response.

**PDP:** Average model response  
**ICE:** Responses for individual sampled students

![Final model PDP and ICE](../reports/reference/figures/final_model_pdp_ice.png)



## Error Rates Differ Across Student Groups

| Audit finding | Result |
|---|---|
| Dropout recall under age 20 | **54.7%** |
| Dropout recall at age 35+ | **87.2%** |
| Gender dropout FPR gap | **7.3 percentage points** |
| Tuition-status dropout FPR gap | **42.1 percentage points** |

For outstanding fees, the **46.7% false-positive rate** represents
**7 errors among only 15 non-dropouts**.

- Race is unavailable; nationality is not a substitute.
- Financial variables are imperfect socioeconomic proxies.
- Disparities do not by themselves establish causal discrimination.
- Prediction timing and financial-feature reliance require investigation.
- Use predictions for supportive human review, not punitive decisions.

**Further mitigation requires fresh independent evaluation.**

## Conclusions

- Full-feature random forest achieved **0.720 test macro F1**.
- Logistic regression was nearly tied in cross-validation.
- Feature selection and PCA did not improve the tested configurations.
- Enrolled classification and unequal group error rates remain limitations.

## Next Research Steps

1. Verify feature timestamps and define a prospective prediction point.
2. Evaluate on later cohorts and, where possible, another institution.
3. Test mitigation strategies using development data.
4. Measure the outcomes of supportive interventions.

**Prediction accuracy does not establish intervention effectiveness.**

## Supporting Materials

Dataset:
https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success

Methods:
https://scikit-learn.org/stable/
https://fairlearn.org/